### 직방의 원룸 매물정보 수집
- 절차
    - 동이름 -> 위도,경도 
    - 위도,경도 -> geohash(점에 해당하는 범위값) 변환
    - geohash(영역) -> 매물 아이디
    - 매물아이디 -> 매물정보

In [2]:
import requests
import pandas as pd
import geohash2

In [12]:
# 1. 동이름 -> 위도,경도
# decoder https://meyerweb.com/eric/tools/dencoder/
addr="수원시 조원동"
url=f"https://apis.zigbang.com/v2/search?leaseYn=N&q={addr}&serviceType=원룸"
response=requests.get(url)
data=response.json()['items'][0]
lat,lng=data["lat"],data["lng"]
lat,lng

(37.301910400390625, 127.01654052734375)

In [15]:
# 2. 위도, 경도 -> geohash
# precistion이 커질수록 영억이 작아짐
geohash=geohash2.encode(lat,lng,precision=5)
geohash

'wydk4'

In [23]:
# 3. geohash -> 매물아이디
url=f"https://apis.zigbang.com/v2/items?deposit_gteq=0\
    &domain=zigbang&geohash={geohash}&needHasNoFiltered=true&rent_gteq=0&sales_type_in=전세|월세&service_type_eq=원룸"
response=requests.get(url)
response

<Response [200]>

In [21]:
items=response.json()['items']
len(items),items[:2]

(1065,
 [{'lat': 37.26589936686827, 'lng': 127.03134791185127, 'item_id': 32830882},
  {'lat': 37.26540533362609, 'lng': 127.03071367633807, 'item_id': 32831244}])

In [ ]:
ids = [item["item_id"] for item in items]
ids[:5]

In [27]:
# 4. 매물아이디 -> 매물정보
url="https://apis.zigbang.com/v2/items/list"
params={
    "domain":"zigbang",
    "withCoalition": "true",
    "item_ids":ids[:900]
}
response=requests.post(url,params)
response

<Response [200]>

In [33]:
items=response.json()["items"]
items[:2]
columns=["item_id","sales_type","deposit","rent","size_m2","manage_cost"]
df=pd.DataFrame(items)[columns]
df.tail(2)

,item_id,sales_type,deposit,rent,size_m2,manage_cost
898,32872674,월세,500,35,23.14,3
899,32704557,전세,4200,0,19.00,0


In [38]:
pd.options.display.max_columns,pd.options.display.max_rows

(50, 60)

In [35]:
# max row, max column 설정
pd.options.display.max_columns=50

In [37]:
pd.DataFrame(items).tail(2)

,section_type,item_id,images_thumbnail,sales_type,sales_title,deposit,rent,size_m2,공급면적,전용면적,계약면적,room_type_title,floor,floor_string,building_floor,title,is_first_movein,room_type,address,random_location,is_zzim,status,service_type,tags,address1,address2,address3,manage_cost,reg_date,is_new
898,None,32872674,https://ic.zigbang.com/ic/items/32872674/1.jpg,월세,월세,500,35,23.14,"{'m2': 23.14, 'p': '7'}","{'m2': 23.14, 'p': '7'}",None,None,1,1,3,🚨오픈형🚨💥아주대/광교역💥베란다 있는 깔끔한집💥,None,01,수원시 팔달구 우만동,"{'lat': 37.28309786382024, 'lng': 127.03136250...",False,True,원룸,[추천],경기도 수원시 팔달구 우만동,None,None,3,2022-08-03T19:16:10+09:00,True
899,None,32704557,https://ic.zigbang.com/ic/items/32704557/1.jpg,전세,전세,4200,0,19.00,"{'m2': 19, 'p': '5.7'}","{'m2': 19, 'p': '5.7'}",None,None,1,1,2,ㅁ 1인가구 원룸 전세,None,01,수원시 팔달구 지동,"{'lat': 37.28387025481651, 'lng': 127.02430960...",False,True,원룸,[],경기도 수원시 팔달구 지동,None,None,0,2022-07-21T13:52:15+09:00,False


#### 함수로 만들기

In [47]:
def oneroom(addr):
    url=f"https://apis.zigbang.com/v2/search?leaseYn=N&q={addr}&serviceType=원룸"
    response=requests.get(url)
    data=response.json()['items'][0]
    lat,lng=data["lat"],data["lng"]
    geohash=geohash2.encode(lat,lng,precision=5)
    
    url=f"https://apis.zigbang.com/v2/items?deposit_gteq=0\
    &domain=zigbang&geohash={geohash}&needHasNoFiltered=true&rent_gteq=0&sales_type_in=전세|월세&service_type_eq=원룸"
    response=requests.get(url)
    items = response.json()["items"]
    ids = [item["item_id"] for item in items]
    
    url="https://apis.zigbang.com/v2/items/list"
    params={
        "domain":"zigbang",
        "withCoalition": "true",
        "item_ids":ids[:900]
    }
    response=requests.post(url,params)
    
    items = response.json()["items"]
    columns = ["item_id", "sales_type", "deposit", "rent", "size_m2", "address1", "manage_cost"]
    return pd.DataFrame(items)[columns]
    

In [51]:
addr="마포구 합정동"
df = oneroom(addr)

In [52]:
df_filtered = df[df["address1"].str.contains(addr)].reset_index(drop=True)
df_filtered

,item_id,sales_type,deposit,rent,size_m2,address1,manage_cost
0,32662713,전세,17700,0,39.67,서울시 마포구 합정동,0
1,32668759,전세,17700,0,39.67,서울시 마포구 합정동,0
2,32726690,전세,19900,0,36.28,서울시 마포구 합정동,0
3,32542700,월세,1500,95,49.59,서울시 마포구 합정동,7
4,32513282,전세,5000,0,23.14,서울시 마포구 합정동,5
...,...,...,...,...,...,...,...
95,32794268,월세,1000,90,59.50,서울시 마포구 합정동,2
96,32794303,월세,3000,80,59.50,서울시 마포구 합정동,2
97,32804404,월세,2000,85,61.24,서울시 마포구 합정동,2
98,32812101,월세,3000,80,61.24,서울시 마포구 합정동,2


### 모듈 파일 만들기 :.py

In [57]:
%%writefile zigbang.py
#패키지 추가필수
import requests
import pandas as pd
import geohash2
def oneroom(addr):
    url=f"https://apis.zigbang.com/v2/search?leaseYn=N&q={addr}&serviceType=원룸"
    response=requests.get(url)
    data=response.json()['items'][0]
    lat,lng=data["lat"],data["lng"]
    geohash=geohash2.encode(lat,lng,precision=5)
    
    url=f"https://apis.zigbang.com/v2/items?deposit_gteq=0\
    &domain=zigbang&geohash={geohash}&needHasNoFiltered=true&rent_gteq=0&sales_type_in=전세|월세&service_type_eq=원룸"
    response=requests.get(url)
    items = response.json()["items"]
    ids = [item["item_id"] for item in items]
    
    url="https://apis.zigbang.com/v2/items/list"
    params={
        "domain":"zigbang",
        "withCoalition": "true",
        "item_ids":ids[:900]
    }
    response=requests.post(url,params)
    
    items = response.json()["items"]
    columns = ["item_id", "sales_type", "deposit", "rent", "size_m2", "address1", "manage_cost"]
    return pd.DataFrame(items)[columns]
    

Overwriting zigbang.py


In [58]:
import zigbang as zb

In [59]:
df=zb.oneroom("망원동")
df.tail(2)

,item_id,sales_type,deposit,rent,size_m2,address1,manage_cost
744,32862490,전세,43000,0,31.74,서울시 마포구 중동,5
745,32866975,월세,3000,90,49.59,서울시 마포구 중동,2


In [61]:
%reset

In [62]:
data=1

In [63]:
#변수목록 보여줌
%whos

Variable   Type    Data/Info
----------------------------
data       int     1
